# Snapshot & Restore a Starting Design

`minimize()` mutates the optic **in place**: after the call, the lens holds the
optimized surface parameters. To preserve the starting design so you can compare
before/after results (or roll back), take a `copy.deepcopy()` **before** calling
`minimize()`.

In [ ]:
import copy

import numpy as np

from optiland import optic
from optiland.optimization import OptimizationProblem, minimize

## Build a simple singlet

In [ ]:
lens = optic.Optic()

lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(index=1, thickness=7, radius=50, material="N-BK7", is_stop=True)
lens.surfaces.add(index=2, thickness=45, radius=-50)
lens.surfaces.add(index=3)

lens.set_aperture(aperture_type="EPD", value=10)
lens.fields.set_type("angle")
lens.fields.add(y=0.0)
lens.wavelengths.add(value=0.5876, is_primary=True)
lens.update_paraxial()

lens.draw(num_rays=5)

## Snapshot the starting design

A `deepcopy` captures the complete state of the optic — surfaces, geometry,
materials, aperture, fields, and wavelengths — so nothing is shared between
`starting` and `lens` after this point.

In [ ]:
starting = copy.deepcopy(lens)

## Define the optimization problem

In [ ]:
problem = OptimizationProblem()

input_data = {
    "optic": lens,
    "surface_number": -1,
    "Hx": 0,
    "Hy": 0,
    "num_rays": 5,
    "wavelength": 0.5876,
    "distribution": "hexapolar",
}
problem.add_operand("rms_spot_size", target=0, weight=1, input_data=input_data)

problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)
problem.add_variable(lens, "thickness", surface_number=2)

## Run optimization

In [ ]:
result = minimize(problem, "dls")
print(result)

## Compare starting vs. optimized design

`lens` now holds the optimized parameters; `starting` is untouched.

In [ ]:
r1_start = starting.surfaces.surfaces[1].geometry.radius
r1_opt   = lens.surfaces.surfaces[1].geometry.radius

r2_start = starting.surfaces.surfaces[2].geometry.radius
r2_opt   = lens.surfaces.surfaces[2].geometry.radius

print(f"Surface 1 radius  — starting: {r1_start:.3f} mm   optimized: {r1_opt:.3f} mm")
print(f"Surface 2 radius  — starting: {r2_start:.3f} mm   optimized: {r2_opt:.3f} mm")

## Draw starting and optimized layouts side by side

In [ ]:
print("Starting design:")
starting.draw(num_rays=5)

print("Optimized design:")
lens.draw(num_rays=5)